In [15]:
# Imports and config

import os
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
from config import load_settings

settings = load_settings()
folder = settings['paths']['folder']
input_path = os.path.join(folder, settings['paths']['input_file'])

AM_PEAK_PERIODS = settings['trip_filter']['am_peak_periods']
CAR_MODES = set(settings['trip_filter']['car_modes'])


In [12]:
# Load and filter data (slow — only re-run if input file changes)

df = pd.read_csv(input_path)
df_filtered = df[
    df['depart_period'].isin(AM_PEAK_PERIODS) &
    df['trip_mode_name'].isin(CAR_MODES)
].copy()

print(f"Total AM peak car trips: {len(df_filtered):,}")


ParserError: Error tokenizing data. C error: Calling read(nbytes) on source failed. Try engine='python'.

In [ ]:
# Count work vs other auto trips

work_tour_mask = df_filtered['tour_purpose'].str.lower().str.startswith(('work_', 'atwork_'))
n_work  = work_tour_mask.sum()
n_other = (~work_tour_mask).sum()
total   = len(df_filtered)

print(f"Total AM peak car trips : {total:>12,}")
print(f"  Work auto trips       : {n_work:>12,}  ({100*n_work/total:.1f}%)")
print(f"  Other auto trips      : {n_other:>12,}  ({100*n_other/total:.1f}%)")
print()
print("Work trips by tour_purpose:")
print(df_filtered.loc[work_tour_mask, 'tour_purpose'].value_counts())

In [ ]:
# Count work vs other auto trips by dest_purpose

total = len(df_filtered)
dest_purpose_counts = df_filtered['dest_purpose'].value_counts()

n_work_dest  = dest_purpose_counts.get('work', 0) + dest_purpose_counts.get('Work', 0)
n_other_dest = total - n_work_dest

print(f"Total AM peak car trips : {total:>12,}")
print(f"  Work dest trips       : {n_work_dest:>12,}  ({100*n_work_dest/total:.1f}%)")
print(f"  Other dest trips      : {n_other_dest:>12,}  ({100*n_other_dest/total:.1f}%)")

In [ ]:
# Count work vs other auto trips in T171 (PECAS) trips
# TripCalculationsI.csv only holds the coefficients that define each TripType;
# the resulting trips (one row per trip, tagged by type) are in tazTrips.csv.

trip_calcs = pd.read_csv(settings['paths']['t171_trip_calculations_i'])
print("TripCalculationsI coefficients by TripType:")
print(trip_calcs.groupby('TripType')['CommodityName'].apply(list).to_string())
print()

t171_trips = pd.read_csv(settings['paths']['t171_2020_taz_trips_csv'], usecols=['type'])
t171_type_counts = t171_trips['type'].value_counts()

t171_total   = len(t171_trips)
t171_n_work  = t171_type_counts.get('work', 0)
t171_n_other = t171_total - t171_n_work

print(f"Total T171 2020 trips   : {t171_total:>12,}")
print(f"  Work trips            : {t171_n_work:>12,}  ({100*t171_n_work/t171_total:.1f}%)")
print(f"  Other trips           : {t171_n_other:>12,}  ({100*t171_n_other/t171_total:.1f}%)")

In [ ]:
# Count total trips in the OMX matrix

import openmatrix as omx
import numpy as np

omx_path = os.path.join(
    folder,
    settings['aequilibrae_network']['project_path'],
    'matrices',
    os.path.splitext(settings['paths']['trip_list_file'])[0] + '.omx'
)

with omx.open_file(omx_path, 'r') as m:
    total_omx = np.array(m['trips']).sum()

print(f"Total trips in OMX matrix: {total_omx:,.1f}")

In [ ]:
# Cell 6: Tabulate counts by dest_purpose

print(df_filtered['dest_purpose'].value_counts())

In [ ]:
# Inspect trips_joint_AM.OMX

import openmatrix as omx
import numpy as np

joint_omx_path = os.path.join(folder, 'trips_joint_AM.OMX')

with omx.open_file(joint_omx_path, 'r') as m:
    print(f"Matrix names : {m.list_matrices()}")
    print(f"Shape        : {m.shape()}")
    for name in m.list_matrices():
        data = np.array(m[name])
        print(f"\n  [{name}]")
        print(f"    sum      : {data.sum():,.4f}")
        print(f"    min      : {data.min()}")
        print(f"    max      : {data.max()}")
        print(f"    nonzero  : {np.count_nonzero(data):,}")

In [ ]:

# Table of trips by matrix in trips_joint_AM.OMX

with omx.open_file(joint_omx_path, 'r') as m:
    rows = [{'matrix': name, 'trips': np.array(m[name]).sum()} for name in m.list_matrices()]

summary = pd.DataFrame(rows).set_index('matrix')
summary['trips'] = summary['trips'].map('{:,.1f}'.format)
print(summary.to_string())